# 台灣身分證字號驗證器

## 身分證字號格式
- 共 **10 碼**
- 第 1 碼:大寫英文字母(地區代碼)
- 第 2 碼:性別碼(1 = 男, 2 = 女)
- 第 3~9 碼:流水號(數字)
- 第 10 碼:檢查碼

## 檢查碼計算規則
1. 將英文字母轉成兩位數字(例如 A=10, B=11, ... , I=34, O=35, W=32)
2. 字母的十位數 × **1**、個位數 × **9**
3. 其餘 9 位數字依序 × **8, 7, 6, 5, 4, 3, 2, 1, 1**
4. 全部相加的總和必須是 **10 的倍數**

In [22]:
# 英文字母對應的地區代碼(數字)
letter_to_num = {
    'A': 10, 'B': 11, 'C': 12, 'D': 13, 'E': 14, 'F': 15,
    'G': 16, 'H': 17, 'I': 34, 'J': 18, 'K': 19, 'L': 20,
    'M': 21, 'N': 22, 'O': 35, 'P': 23, 'Q': 24, 'R': 25,
    'S': 26, 'T': 27, 'U': 28, 'V': 29, 'W': 32, 'X': 30,
    'Y': 31, 'Z': 33
}

def check_id(id_no):
    """檢查台灣身分證字號是否正確"""
    id_no = id_no.strip().upper()

    # 1. 檢查長度是否為 10
    if len(id_no) != 10:
        return False, "錯誤!長度必須為 10 碼,你輸入了 %d 碼(正確格式範例:A123456789)" % len(id_no)

    # 2. 檢查第 1 碼是否為英文字母
    if not id_no[0].isalpha():
        return False, "錯誤!第 1 碼必須是英文字母 A~Z(正確格式範例:A123456789)"

    # 3. 檢查第 2 碼是否為性別碼(1 或 2)
    if id_no[1] not in ('1', '2'):
        return False, "錯誤!第 2 碼必須是性別碼 1(男)或 2(女),你輸入了 %s" % id_no[1]

    # 4. 檢查第 3~10 碼是否為數字
    if not id_no[2:].isdigit():
        return False, "錯誤!第 3~10 碼必須是數字,請檢查是否有英文字母或符號"

    # 5. 計算檢查碼
    # 字母轉成兩位數字,十位數 ×1,個位數 ×9
    code = letter_to_num[id_no[0]]
    total = (code // 10) * 1 + (code % 10) * 9

    # 其餘 9 位數字依序 × 8, 7, 6, 5, 4, 3, 2, 1, 1
    weights = [8, 7, 6, 5, 4, 3, 2, 1, 1]
    for d, w in zip(id_no[1:], weights):
        total += int(d) * w

    # 總和必須是 10 的倍數
    if total % 10 == 0:
        return True, "身分證字號正確"
    else:
        return False, "錯誤!檢查碼不符,此身分證字號不存在"

In [18]:
import os

# 若未設定顯示,自動連到本機桌面(適用於 SSH 遠端執行)
if not os.environ.get("DISPLAY"):
    os.environ["DISPLAY"] = ":0"

import tkinter as tk
from tkinter import messagebox

def validate():
    id_no = entry.get().strip().upper()
    ok, msg = check_id(id_no)
    if ok:
        messagebox.showinfo("驗證結果", f"✓ {msg}\n\n{id_no}")
    else:
        messagebox.showerror("驗證結果", f"✗ {msg}")

# 建立主視窗
root = tk.Tk()
root.title("台灣身分證字號驗證器")
root.geometry("420x230")
root.resizable(False, False)

# 標題與說明
tk.Label(root, text="台灣身分證字號驗證器", font=("Arial", 18, "bold")).pack(pady=10)
tk.Label(root, text="格式:1 個大寫英文字母 + 9 位數字(共 10 碼)", font=("Arial", 11)).pack()

# 輸入框
entry = tk.Entry(root, font=("Arial", 16), justify="center", width=18)
entry.pack(pady=10)

# 按鈕
btn_frame = tk.Frame(root)
btn_frame.pack(pady=5)
tk.Button(btn_frame, text="確認", command=validate, font=("Arial", 13), width=8).pack(side="left", padx=10)
tk.Button(btn_frame, text="離開", command=root.destroy, font=("Arial", 13), width=8).pack(side="left", padx=10)

# 按下 Enter 也可驗證
entry.bind("<Return>", lambda e: validate())
entry.focus_set()

root.mainloop()


TclError: no display name and no $DISPLAY environment variable

In [ ]:
# 測試範例(含正確與各種錯誤,可用來觀察提示訊息)
test_cases = [
    "A123456789",   # 正確
    "B276543214",   # 正確
    "C187654325",   # 正確
    "B123456789",   # 檢查碼錯誤
    "A12345678",    # 長度不足
    "A323456789",   # 性別碼錯誤
    "A12345678A",   # 最後一碼不是數字
]

for case in test_cases:
    ok, msg = check_id(case)
    print(f"{case} -> {msg}")


A123456789 -> 身分證字號正確
B276543214 -> 身分證字號正確
C187654325 -> 身分證字號正確
B123456789 -> 錯誤!檢查碼不符,此身分證字號不存在
A12345678 -> 錯誤!長度必須為 10 碼,你輸入了 9 碼(正確格式範例:A123456789)
A323456789 -> 錯誤!第 2 碼必須是性別碼 1(男)或 2(女),你輸入了 3
A12345678A -> 錯誤!第 3~10 碼必須是數字,請檢查是否有英文字母或符號


# CGI 網頁版驗證器

本機另有 **CGI 文字輸入確認介面**(網頁表單),檔案如下:

- `cgi-bin/id_check.py` — CGI 程式(含驗證邏輯 + HTML 表單)
- `cgi_server.py` — 單執行緒 CGI 啟動腳本

## 啟動方式

```bash
cd 0816
python3 cgi_server.py
```

啟動後開啟瀏覽器連到:

```
http://<本機IP>:8000/cgi-bin/id_check.py
```

輸入身分證字號按「確認」,網頁會顯示 **✓ 正確** 或 **✗ 錯誤原因**。